In [1]:
from dask.distributed import LocalCluster, Client
cluster = LocalCluster(n_workers=4, threads_per_worker=1,  memory_limit='6GB')
client = Client(cluster)

In [2]:
from tropical_cyclones import TCs, multi_plot, plot_trajectories
from aqua.core.util import load_yaml
config = load_yaml('/work/users/mccorda/tc_analysis_csv/config_tcs_clara.yaml')
#config = load_yaml('/work/users/mccorda/AQUA-diagnostics/frontier-diagnostics/tropical_cyclones/config/config_tcs_clara.yaml')

/work/users/mccorda/miniforge3/envs/aqua-diagnostics/lib/python3.12/site-packages/intake_esm/__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [3]:
# initialise tropical class with streaming options
tropical = TCs(tdict=config, streaming=True,
                stream_step=config['stream']['streamstep'],
                stream_startdate=config['time']['startdate'],
                paths=config['paths'],
                loglevel=config['setup']['loglevel'],
                orography=True,
                nproc=1)

2026-05-15 15:44:23 :: TCs :: WARNING  -> Model ERA5 - Exp: era5
2026-05-15 15:44:23 :: TCs :: WARNING  -> Initialised streaming for 1D days starting on 1979-01-01 00:00:00


In [4]:
from glob import glob
import os

# Loop per generare tracce
tropical.loop_streaming(config)
print("[INFO] Rilevamento cicloni tropicali completato.")

    # === GESTIONE FILE TRACCIA ===
trackdir = config['paths']['trackdir']
tmpdir = config['paths']['tmpdir']

    # Mostra i file trovati
filenames = sorted(glob(os.path.join(trackdir, "tempest_track*.txt")))
print(f"[DEBUG] File di traccia trovati:\n  {filenames}\nTotale: {len(filenames)}")

    # Percorso file totale con data
tot_file = os.path.join(
        tmpdir,
        f"tempest_tracks_tot_{config['time']['startdate']}_{config['time']['enddate']}.txt"
)
print(f"[DEBUG] Percorso file totale: {tot_file}")

    # Scrittura file totale
os.makedirs(tmpdir, exist_ok=True)
with open(tot_file, 'w') as output_file:
        for fname in filenames:
            print(f"[DEBUG] Aggiungo: {fname}")
            with open(fname) as infile:
                for line in infile:
                    output_file.write(line)

2026-05-15 15:45:42 :: TCs :: WARNING  -> processing time step 19790101T00
/work/users/mccorda/miniforge3/envs/aqua-diagnostics/lib/python3.12/site-packages/distributed/client.py:3374: UserWarning: Sending large graph of size 49.38 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
/work/users/mccorda/miniforge3/envs/aqua-diagnostics/lib/python3.12/site-packages/distributed/client.py:3374: UserWarning: Sending large graph of size 46.40 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
/work/users/mccorda/miniforge3/

[INFO] Rilevamento cicloni tropicali completato.
[DEBUG] File di traccia trovati:
  []
Totale: 0
[DEBUG] Percorso file totale: /work/users/mccorda/tc_analysis_csv/tmpdir/tempest_tracks_tot_19790101_19790105.txt


In [5]:
!which DetectNodes

/work/users/mccorda/miniforge3/envs/aqua-diagnostics/bin/DetectNodes


In [9]:
# ===========================================
# 1. SETUP - Parametri da config
# ===========================================
import os
from glob import glob
from datetime import datetime

trackdir = config['paths']['trackdir']
tmpdir = config['paths']['tmpdir']
era5_dir = "/work/users/mccorda/data"
start_date = datetime.strptime(config['time']['startdate'], "%Y%m%d")
end_date   = datetime.strptime(config['time']['enddate'], "%Y%m%d")

In [10]:
# 2. SELEZIONE FILE - Trova i file tempest_track nel range di date
# ===========================================
all_files = sorted(glob(os.path.join(trackdir, "tempest_track_*.txt")))
filenames = []
for f in all_files:
    base = os.path.basename(f)
    try:
        parts = base.replace("tempest_track_", "").replace(".txt", "").split("-")
        start_f = datetime.strptime(parts[0], "%Y%m%d")
        end_f = datetime.strptime(parts[1], "%Y%m%d")
        if end_f >= start_date and start_f <= end_date:
            filenames.append(f)
    except:
        pass

print(f"[INFO] File selezionati: {len(filenames)}")

[INFO] File selezionati: 0


In [4]:
import xarray as xr
xr.open_dataset("/work/users/mccorda/data/orog_ERA5.nc", engine="netcdf4")

<xarray.Dataset> Size: 8MB
Dimensions:    (time: 1, latitude: 721, longitude: 1440)
Coordinates:
  * time       (time) datetime64[ns] 8B 1979-12-15
  * longitude  (longitude) float32 6kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
  * latitude   (latitude) float32 3kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
Data variables:
    z          (time, latitude, longitude) float64 8MB ...
Attributes:
    CDI:          Climate Data Interface version 1.9.7.1 (http://mpimet.mpg.d...
    Conventions:  CF-1.6
    history:      Wed Apr 01 09:15:38 2020: cdo selmon,12 orog_ERA5.nc orog_E...
    CDO:          Climate Data Operators version 1.9.7.1 (http://mpimet.mpg.d...